# Training and Inference

This notebook separates training from inference: predicting on unseen instances, thresholding probabilities, persisting a model with joblib and showing why a scaler must not be re-fitted at inference time.

## Import Required Libraries

Import joblib, a temporary-file helper and the scikit-learn tools.

In [1]:
# Import the required libraries.
import os
import tempfile

import numpy as np
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

## Train a Model

Fit a scaled logistic regression on the breast cancer training set; this is the training phase.

In [ ]:
# Load the data and train a scaled logistic regression.
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

model = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=5000))
model.fit(X_train, y_train)
print(f"Training accuracy: {model.score(X_train, y_train):.3f}")

Training accuracy: 0.987


## Inference on New Unseen Instances

Inference reuses the fitted model to predict labels for instances it never saw during training.

In [3]:
# Predict on brand new unseen instances from the test set.
new_instances = X_test[:5]
preds = model.predict(new_instances)
print(f"Predicted labels: {preds}")
print(f"True labels:      {y_test[:5]}")

Predicted labels: [0 1 1 0 0]
True labels:      [0 1 1 0 0]


## predict vs predict_proba and Thresholding

`predict` uses a default 0.5 cut-off; changing the threshold on `predict_proba` trades precision for recall.

In [4]:
# Compare default predictions with a custom threshold.
proba = model.predict_proba(X_test)[:, 1]
default_pred = (proba >= 0.5).astype(int)
strict_pred = (proba >= 0.8).astype(int)
print(f"Positives at 0.5 threshold: {default_pred.sum()}")
print(f"Positives at 0.8 threshold: {strict_pred.sum()}")

Positives at 0.5 threshold: 107
Positives at 0.8 threshold: 102


## Persisting and Reloading the Model

A fitted model is saved to disk with joblib and reloaded so it can serve predictions without retraining.

In [5]:
# Save the fitted model to a temporary file and reload it.
tmp_path = os.path.join(tempfile.mkdtemp(), "model.joblib")
joblib.dump(model, tmp_path)
reloaded = joblib.load(tmp_path)

# The reloaded model reproduces the original predictions.
same = np.array_equal(model.predict(X_test),
                      reloaded.predict(X_test))
print(f"Saved to: {tmp_path}")
print(f"Reloaded predictions identical: {same}")

Saved to: /var/folders/q3/2tkw4wl936l357n57y8sx0x80000gn/T/tmpp8e3r85v/model.joblib
Reloaded predictions identical: True


## Why You Must Not Re-fit a Scaler at Inference

Re-fitting a scaler on the test data uses different statistics than training, silently corrupting the predictions.

In [6]:
# Fit a scaler correctly on the training data only.
scaler = StandardScaler().fit(X_train)
clf = LogisticRegression(max_iter=5000).fit(
    scaler.transform(X_train), y_train)
correct = clf.score(scaler.transform(X_test), y_test)

# Wrongly re-fit a fresh scaler on the test data.
wrong_scaler = StandardScaler().fit(X_test)
wrong = clf.score(wrong_scaler.transform(X_test), y_test)

print(f"Accuracy with training scaler:  {correct:.3f}")
print(f"Accuracy re-fitting at inference: {wrong:.3f}")

Accuracy with training scaler:  0.988
Accuracy re-fitting at inference: 0.965


## Summary

- **Training**: fit parameters once on the training data.
- **Inference**: reuse the fitted model on unseen instances.
- **Thresholding**: `predict_proba` lets you move the decision cut-off away from 0.5.
- **Persistence**: `joblib` saves and reloads a fitted model exactly.
- **No re-fitting**: transformers must be fit on training data and only applied at inference.